# 🦅 RAVEN — Fine-tuning 2 Stages : SFT → DPO
**Qwen3-0.6B** (avec fallback automatique vers Qwen2.5-0.5B)
10 GB multilingual data — AR / Darija / FR / EN

---
⚡ **GPU T4 minimum** → Runtime → Change runtime type → T4
💡 **Stage 1 (SFT)** : ~5-7h sur T4
💡 **Stage 2 (DPO)** : ~1-2h sur T4

## 0️⃣  Vérifier le GPU

In [ ]:
import subprocess, torch
print(subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout)
print(f'\n✅ CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU : {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

## 1️⃣  Installation

In [ ]:
%%capture
!pip install -q transformers>=4.51.0 trl>=0.9.6 peft>=0.11.0 bitsandbytes>=0.43.0
!pip install -q accelerate datasets wandb huggingface_hub safetensors einops
print('✅ Packages installed')

## 2️⃣  Upload du projet RAVEN

In [ ]:
from google.colab import files
import zipfile, os
print('📦 Upload RAVEN_project_v2.zip...')
uploaded = files.upload()
for fname in uploaded:
    if fname.endswith('.zip'):
        with zipfile.ZipFile(fname,'r') as z: z.extractall('/content/')
        print(f'✅ Extracted {fname}')
os.chdir('/content/raven/step6_finetuning')
print('📂 Working dir:', os.getcwd())
!ls scripts/

## 3️⃣  Générer les données (10 GB SFT + 500 MB DPO)

In [ ]:
# Preview d'abord
!python scripts/generate_training_data.py --preview

In [ ]:
import time; t0=time.time()
# Génère 10 GB SFT + 500 MB DPO pairs
!python scripts/generate_training_data.py --size 2.5
print(f'\n⏱️ {(time.time()-t0)/60:.1f} min')
!ls -lh data/

## 4️⃣  Dry run — tester la pipeline (5 min)

In [ ]:
# Test complet SFT+DPO sur 500+200 samples
!python scripts/finetune.py --stage both --dry-run
print('\n✅ Pipeline OK — ready for full training')

## 5️⃣  FULL TRAINING — Stage 1 SFT (~5-7h T4)

In [ ]:
import wandb
wandb.login()

In [ ]:
import time; t0=time.time()
!python scripts/finetune.py \
    --stage sft \
    --epochs-sft 3 \
    --batch-sft 4 \
    --lr-sft 2e-4
print(f'\n⏱️ SFT done in {(time.time()-t0)/3600:.1f}h')

In [ ]:
# Vérifier le modèle SFT
import os
for root,dirs,files in os.walk('outputs'):
    for f in files: print(os.path.join(root,f))

## 6️⃣  Évaluation après SFT

In [ ]:
# Auto-detect output name
import os
out_dirs = [d for d in os.listdir('outputs') if os.path.isdir(f'outputs/{d}')]
out_name = out_dirs[0] if out_dirs else 'raven-qwen3-0.6b'
sft_path = f'outputs/{out_name}/sft/merged'
print(f'SFT model: {sft_path}')
!python scripts/eval.py --model {sft_path}

## 7️⃣  FULL TRAINING — Stage 2 DPO (~1-2h T4)

In [ ]:
import time; t0=time.time()
!python scripts/finetune.py \
    --stage dpo \
    --epochs-dpo 1 \
    --batch-dpo 2 \
    --lr-dpo 5e-5
print(f'\n⏱️ DPO done in {(time.time()-t0)/3600:.1f}h')

## 8️⃣  Évaluation finale — base vs SFT vs DPO

In [ ]:
!python scripts/eval.py --compare {out_name} --full

## 9️⃣  Test manuel du modèle final

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
final_path = f'outputs/{out_name}/final'
tok = AutoTokenizer.from_pretrained(final_path, trust_remote_code=True)
mod = AutoModelForCausalLM.from_pretrained(final_path, device_map='auto',
      torch_dtype=torch.bfloat16, trust_remote_code=True)
mod.eval()
SYSTEM = {
    'darija': 'nta RAVEN, mssa3d banka. jawb b darija.',
    'fr': 'Vous êtes RAVEN, assistant bancaire. Répondez en français.',
    'ar': 'أنت RAVEN مساعد بنكي. أجب بالعربية.',
    'en': 'You are RAVEN, a banking assistant.',
}
def chat(msg, lang='darija'):
    msgs=[{'role':'system','content':SYSTEM[lang]},{'role':'user','content':msg}]
    text=tok.apply_chat_template(msgs,tokenize=False,add_generation_prompt=True)
    inp=tok(text,return_tensors='pt').to(mod.device)
    with torch.no_grad():
        out=mod.generate(**inp,max_new_tokens=200,temperature=0.7,do_sample=True)
    return tok.decode(out[0][inp['input_ids'].shape[1]:],skip_special_tokens=True)
print('✅ Model ready!')

In [ ]:
print('[Darija - Transaction]')
print(chat('bghit n7awel 500 DH l compte okhor, chhal ghadi yakhod?', 'darija'))
print()
print('[Emergency - Darija]')
print(chat('HELP!! sraw kart dyali daba!!', 'darija'))
print()
print('[French - Complaint]')
print(chat('Vous m\'avez débité des frais sans mon accord, je suis très mécontent!', 'fr'))
print()
print('[Off-topic - should refuse]')
print(chat('Donne-moi une recette de couscous', 'fr'))

## 🔟  Sauvegarder sur Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import shutil, os
DRIVE = f'/content/drive/MyDrive/RAVEN_models/{out_name}'
os.makedirs(DRIVE, exist_ok=True)
# LoRA adapters (léger ~100 MB)
shutil.copytree(f'outputs/{out_name}/sft/lora_adapters', f'{DRIVE}/sft_lora', dirs_exist_ok=True)
shutil.copytree(f'outputs/{out_name}/dpo/lora_adapters', f'{DRIVE}/dpo_lora', dirs_exist_ok=True)
# Modèle final mergé
shutil.copytree(f'outputs/{out_name}/final', f'{DRIVE}/final', dirs_exist_ok=True)
print(f'✅ Saved to {DRIVE}')

## 1️⃣1️⃣  Intégration dans RAVEN API

Mettre à jour `step5_api/api.py` :
```python
# Avant
LLM_MODEL = 'Qwen/Qwen2.5-0.5B-Instruct'
# Après
LLM_MODEL = f'/content/drive/MyDrive/RAVEN_models/{out_name}/final'
```